In [11]:
import pandas as pd
import os

# =========================
# 1. LOAD TRAIN AUDIO METADATA
# =========================
df = pd.read_csv("train.csv")

# Convert filenames (.ogg → .wav)
df["filename"] = df["filename"].str.replace(".ogg", ".wav", regex=False)

# Path to your converted wav files
bird_audio_dir = "/home/users/ss1482/sangcs372final/Finalproject/birdtrain_wav"

# Build full filepaths
df["filepath"] = df["filename"].apply(lambda x: os.path.join(bird_audio_dir, x))

# Remove missing files (important)
df = df[df["filepath"].apply(os.path.exists)].reset_index(drop=True)

print("Train audio files:", len(df))


# =========================
# 2. LOAD SOUNDSCAPE LABELS
# =========================
ss_df = pd.read_csv("train_soundscapes_labels.csv")

# Convert filenames
ss_df["filename"] = ss_df["filename"].str.replace(".ogg", ".wav", regex=False)

# Path to soundscape wavs
soundscape_dir = "/home/users/ss1482/sangcs372final/Finalproject/train_soundscapes_wav"

# Build filepaths
ss_df["filepath"] = ss_df["filename"].apply(lambda x: os.path.join(soundscape_dir, x))

# Remove missing files
ss_df = ss_df[ss_df["filepath"].apply(os.path.exists)].reset_index(drop=True)

print("Soundscape segments:", len(ss_df))


# =========================
# 3. LOAD TAXONOMY (CRITICAL)
# =========================
taxonomy = pd.read_csv("taxonomy.csv")

labels = taxonomy["primary_label"].values

label_to_idx = {label: i for i, label in enumerate(labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

NUM_CLASSES = len(labels)

print("Total classes:", NUM_CLASSES)  # should be 234


# =========================
# 4. (OPTIONAL) SANITY CHECK
# =========================
print("Unique train_audio species:", df["primary_label"].nunique())
print("Species missing from train_audio:", NUM_CLASSES - df["primary_label"].nunique())

Train audio files: 0
Soundscape segments: 0
Total classes: 234
Unique train_audio species: 0
Species missing from train_audio: 234


In [12]:
from torch.utils.data import DataLoader
bird_dataset = BirdDataset(df, label_to_idx)
ss_dataset   = SoundscapeDataset(ss_df, label_to_idx)

In [13]:
import numpy as np
import soundfile as sf
import librosa

TARGET_SR = 32000

def load_audio_segment(filepath, target_len, start_sec=None):
    try:
        audio, sr = sf.read(filepath)
    except:
        return np.zeros(target_len, dtype=np.float32)

    # mono
    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)

    # resample
    if sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)

    audio = audio.astype(np.float32)

    # if start provided → slice
    if start_sec is not None:
        start = int(start_sec * TARGET_SR)
        end = start + target_len
        audio = audio[start:end]

    # pad / trim
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    return audio



In [6]:
import torch
from torch.utils.data import Dataset

class BaseAudioDataset(Dataset):
    def __init__(self, df, label_to_idx, clip_duration, multi_label=False):
        self.df = df.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.clip_len = TARGET_SR * clip_duration
        self.multi_label = multi_label

    def __len__(self):
        return len(self.df)

    def get_audio(self, row):
        # default: full clip (train_audio)
        return load_audio_segment(
            row["filepath"],
            self.clip_len
        )

    def get_label(self, row):
        label = torch.zeros(len(self.label_to_idx))

        if self.multi_label:
            species_list = str(row["primary_label"]).split(";")
            for sp in species_list:
                if sp in self.label_to_idx:
                    label[self.label_to_idx[sp]] = 1.0
        else:
            label_idx = self.label_to_idx[row["primary_label"]]
            label[label_idx] = 1.0

        return label

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        audio = self.get_audio(row)
        audio = torch.tensor(audio).unsqueeze(0)

        label = self.get_label(row)

        return audio, label

In [7]:
class BirdDataset(BaseAudioDataset):
    def __init__(self, df, label_to_idx):
        super().__init__(
            df,
            label_to_idx,
            clip_duration=10,
            multi_label=False
        )


In [8]:
class SoundscapeDataset(BaseAudioDataset):
    def __init__(self, df, label_to_idx):
        super().__init__(
            df,
            label_to_idx,
            clip_duration=5,
            multi_label=True
        )

    def get_audio(self, row):
        return load_audio_segment(
            row["filepath"],
            self.clip_len,
            start_sec=row["start"]
        )

torch.Size([16, 320000])
torch.Size([16, 206])
